# Module 3.6 — Evaluating Generative Output
**DeskMate SLM Curriculum · Phase 3 · Notebook 20**

Four-layer eval harness: reference metrics → task metrics → LLM-as-judge → regression gate.

| Layer | Metrics | Cost |
|---|---|---|
| 1 Reference-based | ROUGE-L, token F1 | Free |
| 2 Task-specific | Sentence count, intent EM | Free |
| 3 LLM-as-judge | Absolute (1-5), pairwise | Claude API |
| 4 Regression | 30 fixed tests, pass/fail | Free |

Read `3.6_gen_eval.md` for full theory.

---

## Step 0 — Install

In [ ]:
%%capture
!pip install -q rouge-score==0.1.2 anthropic transformers==4.44.0 \
               peft==0.12.0 bitsandbytes==0.43.3 torch==2.3.1 \
               datasets==2.21.0 accelerate==0.34.0

In [ ]:
import re, json, pathlib, time, random, os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_from_disk, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from rouge_score import rouge_scorer as rs

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
HAS_GPU = device == 'cuda'
print(f'Device: {device}')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS    = PROJECT_ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
print(f'Runtime: {RUNTIME}')

## Step 2 — Load Base and Tuned Models

In [ ]:
MODEL_NAME   = 'Qwen/Qwen2.5-1.5B'
ADAPTER_PATH = MODELS_DIR / 'deskmate-qlora-adapter'
SYSTEM_MSG   = 'You are DeskMate, a concise support assistant. Respond in 2-4 sentences.'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_cfg = None
if HAS_GPU:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16)

print('Loading base model ...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_cfg if HAS_GPU else None,
    torch_dtype=torch.bfloat16 if HAS_GPU else torch.float32,
    device_map='auto' if HAS_GPU else None,
    trust_remote_code=True,
)
base_model.eval()
print('Base model loaded.')

if ADAPTER_PATH.exists():
    print('Loading tuned model (LoRA adapter) ...')
    tuned_model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
    tuned_model.eval()
    print('Tuned model loaded.')
else:
    print('Adapter not found — using base model as tuned proxy (run Module 3.4 first).')
    tuned_model = base_model

## Step 3 — Load Gold Eval Set

In [ ]:
gold_path = DATA_PROC / 'gold.jsonl'

if gold_path.exists():
    gold_df = pd.read_json(gold_path, lines=True).head(50)
    if 'reference_reply' not in gold_df.columns:
        gold_df['reference_reply'] = gold_df.get('reply', gold_df.get('text', ''))
else:
    print('gold.jsonl not found — building illustrative examples')
    rows = [
        {'ticket': 'I cannot log in — password reset email never arrived.',
         'intent': 'account_access',
         'reference_reply': 'We apologise for the inconvenience. Please check your '
                            'spam folder first, then use the live chat button to verify '
                            'your identity and reset your password manually.'},
        {'ticket': 'I was charged twice for the Pro plan last month.',
         'intent': 'billing_dispute',
         'reference_reply': 'Thank you for flagging this. I have raised a duplicate-charge '
                            'refund request on your account. You should see the credit '
                            'within 3-5 business days.'},
        {'ticket': 'CSV export button throws a 500 error on every browser.',
         'intent': 'technical_bug',
         'reference_reply': 'We have identified the issue with the CSV export feature '
                            'and our engineering team is working on a fix. '
                            'Expected resolution: within 24 hours. We will email you '
                            'when it is resolved.'},
        {'ticket': 'How do I add a team member to my workspace?',
         'intent': 'usage_question',
         'reference_reply': 'To add a team member, go to Settings > Team > Invite member '
                            'and enter their email. They will receive an invitation within '
                            'a few minutes.'},
        {'ticket': 'Your service has been down for two hours. This is unacceptable.',
         'intent': 'outage_report',
         'reference_reply': 'We are aware of the ongoing service disruption and sincerely '
                            'apologise for the impact. Our team is working to restore '
                            'service as quickly as possible. Current ETA: 30 minutes.'},
    ] * 10   # 50 examples
    gold_df = pd.DataFrame(rows[:50])

print(f'Gold eval set: {len(gold_df)} examples')
print(f'Columns: {list(gold_df.columns)}')

## Step 4 — Generate Replies (Base & Tuned)

In [ ]:
def generate_reply(model, tok, ticket, context=None, max_new_tokens=150):
    user   = (('Context: ' + context + '\n\n') if context else '') + 'Ticket: ' + ticket
    msgs   = [{'role':'system','content':SYSTEM_MSG}, {'role':'user','content':user}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=512)
    inputs = {k: v.to(model.device if hasattr(model,'device') else device)
              for k, v in inputs.items()}
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id)
    new_toks = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()

base_replies  = []
tuned_replies = []

for _, row in gold_df.iterrows():
    ticket  = row.get('ticket', row.get('text', ''))
    context = row.get('context')
    base_replies.append(generate_reply(base_model,  tokenizer, ticket, context))
    tuned_replies.append(generate_reply(tuned_model, tokenizer, ticket, context))

print(f'Generated {len(base_replies)} base replies and {len(tuned_replies)} tuned replies.')
print()
# Sample
print('Example:')
print('Ticket :', gold_df.iloc[0].get('ticket', ''))
print('Base   :', base_replies[0])
print('Tuned  :', tuned_replies[0])

## Step 5 — Layer 1: Reference Metrics (ROUGE-L, Token F1)

In [ ]:
rouge = rs.RougeScorer(['rougeL'], use_stemmer=True)

def token_f1(ref, pred):
    ref_toks  = set(re.findall(r'\w+', ref.lower()))
    pred_toks = set(re.findall(r'\w+', pred.lower()))
    if not pred_toks or not ref_toks: return 0.0
    prec = len(ref_toks & pred_toks) / len(pred_toks)
    rec  = len(ref_toks & pred_toks) / len(ref_toks)
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

references = gold_df['reference_reply'].tolist()

base_rouge  = [rouge.score(r, p)['rougeL'].fmeasure for r, p in zip(references, base_replies)]
tuned_rouge = [rouge.score(r, p)['rougeL'].fmeasure for r, p in zip(references, tuned_replies)]
base_f1     = [token_f1(r, p) for r, p in zip(references, base_replies)]
tuned_f1    = [token_f1(r, p) for r, p in zip(references, tuned_replies)]

print(f'ROUGE-L : base={np.mean(base_rouge):.3f}  tuned={np.mean(tuned_rouge):.3f}')
print(f'Token F1: base={np.mean(base_f1):.3f}  tuned={np.mean(tuned_f1):.3f}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (base_s, tuned_s, title) in zip(
    axes,
    [(base_rouge, tuned_rouge, 'ROUGE-L'), (base_f1, tuned_f1, 'Token F1')]):
    ax.hist(base_s,  bins=15, alpha=0.6, label='Base')
    ax.hist(tuned_s, bins=15, alpha=0.6, label='Tuned')
    ax.set_title(title); ax.set_xlabel('Score'); ax.legend()
plt.tight_layout(); plt.show()

## Step 6 — Layer 2: Task Metrics (Conciseness, Intent EM)

In [ ]:
def sentence_count(text):
    return len(re.findall(r'[.!?]+', text.strip())) or 1

def within_spec(reply):
    return 2 <= sentence_count(reply) <= 4

base_within  = [within_spec(r) for r in base_replies]
tuned_within = [within_spec(r) for r in tuned_replies]
base_lens    = [len(r.split()) for r in base_replies]
tuned_lens   = [len(r.split()) for r in tuned_replies]

print(f'Within 2-4 sentences: base={sum(base_within)/len(base_within)*100:.0f}%  '
       f'tuned={sum(tuned_within)/len(tuned_within)*100:.0f}%')
print(f'Mean word count     : base={np.mean(base_lens):.0f}  tuned={np.mean(tuned_lens):.0f}')

# Intent exact-match (if gold has intent labels)
if 'intent' in gold_df.columns:
    gold_intents = gold_df['intent'].tolist()
    INTENTS = ['account_access','billing_dispute','technical_bug',
               'usage_question','outage_report']
    def extract_intent(reply):
        reply_lower = reply.lower()
        for intent in INTENTS:
            kw = intent.replace('_', ' ')
            if kw in reply_lower or intent in reply_lower:
                return intent
        return 'unknown'
    base_intent_em  = [extract_intent(r) == g
                       for r, g in zip(base_replies, gold_intents)]
    tuned_intent_em = [extract_intent(r) == g
                       for r, g in zip(tuned_replies, gold_intents)]
    print(f'Intent EM: base={sum(base_intent_em)/len(base_intent_em)*100:.0f}%  '
           f'tuned={sum(tuned_intent_em)/len(tuned_intent_em)*100:.0f}%')
else:
    print('Intent EM: N/A (no intent column in gold set)')

## Step 7 — Layer 3a: LLM-as-Judge (Absolute Scoring)

Set `RUN_JUDGE = True` and provide `ANTHROPIC_API_KEY` in Colab Secrets.

In [ ]:
RUN_JUDGE = False   # Set True to run LLM-as-judge

def get_api_key():
    key = os.environ.get('ANTHROPIC_API_KEY', '')
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get('ANTHROPIC_API_KEY')
        except Exception:
            pass
    return key

ABSOLUTE_RUBRIC = (
    'Score this customer support reply on a scale of 1-5. '
    '1=unhelpful or wrong, 3=acceptable, 5=excellent. '
    'A shorter complete reply is better than a longer padded reply. '
    'Consider: accuracy, helpfulness, conciseness, tone. '
    'Reply with a SINGLE INTEGER and nothing else.'
    '\n\nTicket: {ticket}\nReply: {reply}\nScore:'
)

base_judge_scores  = []
tuned_judge_scores = []

if RUN_JUDGE:
    import anthropic
    client = anthropic.Anthropic(api_key=get_api_key())
    JUDGE_MODEL = 'claude-haiku-4-5-20251001'
    N_JUDGE = min(20, len(gold_df))
    for i in range(N_JUDGE):
        ticket = gold_df.iloc[i].get('ticket', '')
        for replies_list, scores_list in [
            (base_replies, base_judge_scores),
            (tuned_replies, tuned_judge_scores)
        ]:
            prompt = ABSOLUTE_RUBRIC.format(ticket=ticket, reply=replies_list[i])
            resp   = client.messages.create(
                model=JUDGE_MODEL, max_tokens=5,
                messages=[{'role':'user','content':prompt}]
            )
            raw = resp.content[0].text.strip()
            try:
                scores_list.append(int(re.search(r'[1-5]', raw).group()))
            except Exception:
                scores_list.append(3)
    print(f'Absolute judge ({N_JUDGE} examples):')
    print(f'  Base  mean: {np.mean(base_judge_scores):.2f}')
    print(f'  Tuned mean: {np.mean(tuned_judge_scores):.2f}')
else:
    print('Judge skipped. Set RUN_JUDGE=True to run.')

## Step 8 — Layer 3b: Pairwise Judge with Position-Bias Mitigation

Both orderings run per pair. Only agreements count as a verdict — disagreements → TIE.

In [ ]:
pairwise_results = []

def judge_pair_debiased(ticket, reply_base, reply_tuned, client, model='claude-haiku-4-5-20251001'):
    PROMPT = (
        'You are evaluating customer support replies. '
        'Which reply is better for this ticket? '
        'A shorter complete reply beats a longer padded one. '
        'Reply with exactly one word: A, B, or TIE.'
        '\n\nTicket: {ticket}\nReply A: {a}\nReply B: {b}\nVerdict:'
    )
    def call(a, b):
        resp = client.messages.create(
            model=model, max_tokens=5,
            messages=[{'role':'user','content':
                       PROMPT.format(ticket=ticket, a=a, b=b)}])
        v = resp.content[0].text.strip().upper()
        return v if v in ('A','B','TIE') else 'TIE'
    # Ordering 1: base=A, tuned=B
    v1 = call(reply_base, reply_tuned)
    # Ordering 2: tuned=A, base=B
    v2 = call(reply_tuned, reply_base)
    # Unflip v2: A-win means tuned won in ordering 2
    if v2 == 'A': v2 = 'B'   # tuned won → tuned=B in ordering 1
    elif v2 == 'B': v2 = 'A' # base won
    # Agreement?
    if v1 == v2 and v1 != 'TIE':
        return 'BASE' if v1 == 'A' else 'TUNED'
    return 'TIE'

if RUN_JUDGE:
    wins_base = wins_tuned = ties = 0
    N_PAIR = min(20, len(gold_df))
    for i in range(N_PAIR):
        ticket  = gold_df.iloc[i].get('ticket', '')
        verdict = judge_pair_debiased(
            ticket, base_replies[i], tuned_replies[i], client)
        pairwise_results.append({'idx': i, 'verdict': verdict})
        if verdict == 'BASE':  wins_base  += 1
        elif verdict == 'TUNED': wins_tuned += 1
        else: ties += 1
    print(f'Pairwise judge ({N_PAIR} pairs, both orderings):')
    print(f'  Base wins : {wins_base} ({wins_base/N_PAIR*100:.0f}%)')
    print(f'  Tuned wins: {wins_tuned} ({wins_tuned/N_PAIR*100:.0f}%)')
    print(f'  Ties      : {ties} ({ties/N_PAIR*100:.0f}%)')
else:
    print('Pairwise judge skipped (RUN_JUDGE=False).')

## Step 9 — Position Bias Calibration

Feed 5 pairs where we know the ground-truth winner. Measure how often the judge agrees.

In [ ]:
CALIBRATION_PAIRS = [
    {
        'ticket': 'I cannot log in.',
        'good':   'We are sorry to hear that. Please use the password reset link '
                  'on the login page or contact our support team for manual verification.',
        'bad':    'Login issue. OK.',
        'winner': 'good'
    },
    {
        'ticket': 'Charged twice for Pro plan.',
        'good':   'We have confirmed a duplicate charge on your account and will '
                  'process a full refund within 3-5 business days.',
        'bad':    'Payment error identified.',
        'winner': 'good'
    },
    {
        'ticket': 'Export button broken.',
        'good':   'Our team has identified the issue with the export feature. '
                  'A fix is expected within 24 hours and we will notify you by email.',
        'bad':    'Export is broken. We will fix it.',
        'winner': 'good'
    },
    {
        'ticket': 'How do I invite a teammate?',
        'good':   'Go to Settings > Team > Invite Member and enter their email address.',
        'bad':    'Please visit our help centre for documentation on team features, '
                  'collaboration options, and workspace settings where you will find '
                  'detailed step-by-step guides on all platform features.',
        'winner': 'good'   # shorter is better here
    },
    {
        'ticket': 'Service is down.',
        'good':   'We are aware of the outage and our engineering team is actively '
                  'working on a fix. Estimated resolution: 30 minutes.',
        'bad':    'Sorry for the inconvenience.',
        'winner': 'good'
    },
]

if RUN_JUDGE:
    correct = 0
    print('Position bias calibration (both orderings):')
    for pair in CALIBRATION_PAIRS:
        v = judge_pair_debiased(pair['ticket'], pair['good'], pair['bad'], client)
        # in ordering 1: good=A, bad=B; winner='good' means 'TUNED' in our mapping
        predicted_winner = 'good' if v == 'BASE' else ('bad' if v == 'TUNED' else 'TIE')
        ok = predicted_winner == pair['winner']
        if ok: correct += 1
        print(f'  [{"OK" if ok else "FAIL"}] verdict={v} expected={pair["winner"]}')
    print(f'Calibration accuracy: {correct}/{len(CALIBRATION_PAIRS)}')
else:
    print('Calibration skipped (RUN_JUDGE=False).')

## Step 10 — Hallucination Detection

In [ ]:
def flag_hallucinations(ticket, reply, context=''):
    source          = ticket + ' ' + context
    source_entities = set(re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', source))
    reply_entities  = re.findall(r'\b[A-Z][a-z]+(?:\s[A-Z][a-z]+)*\b', reply)
    # Filter out common words that aren't domain entities
    stop = {'We','Our','Your','Please','Thank','The','I','You','This','That',
            'A','An','In','On','At','To','For','Of','It','Is','Are','Was','Were',
            'DeskMate','Pro'}
    hallucinated = [e for e in reply_entities
                    if e not in source_entities and e not in stop]
    return list(set(hallucinated))

base_halluc  = [flag_hallucinations(gold_df.iloc[i].get('ticket',''), base_replies[i])
                for i in range(len(base_replies))]
tuned_halluc = [flag_hallucinations(gold_df.iloc[i].get('ticket',''), tuned_replies[i])
                for i in range(len(tuned_replies))]

base_halluc_rate  = sum(1 for h in base_halluc  if h) / len(base_halluc)
tuned_halluc_rate = sum(1 for h in tuned_halluc if h) / len(tuned_halluc)
print(f'Hallucination rate (n-gram): '
       f'base={base_halluc_rate*100:.0f}%  tuned={tuned_halluc_rate*100:.0f}%')

# Show examples
print('\nSample flagged entities (base):')
for i, h in enumerate(base_halluc[:10]):
    if h:
        print(f'  ex {i}: {h}')

## Step 11 — Regression Test Suite (30 Fixed Tests)

In [ ]:
REGRESSION_TESTS = [
    {'ticket': 'My account is locked after too many login attempts.',
     'checks': [
         lambda r: 'account' in r.lower(),
         lambda r: any(w in r.lower() for w in ['unlock','reset','contact','support','verify']),
         lambda r: sentence_count(r) <= 6,
         lambda r: 'password' not in r.lower() or any(
             w in r.lower() for w in ['reset','new','change'])]},
    {'ticket': 'I was charged twice for my subscription.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['refund','credit','charge','billing']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'The export button shows a 500 error.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['error','issue','bug','fix','team','engineering']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'How do I invite a colleague to my workspace?',
     'checks': [
         lambda r: any(w in r.lower() for w in ['settings','invite','email','team']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'Your entire service has been down for 3 hours.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['aware','outage','team','investigating','fix']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'I need to cancel my subscription.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['cancel','subscription','account','settings']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'I cannot upload files larger than 10MB.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['upload','file','limit','size','storage']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'Where is my invoice for last month?',
     'checks': [
         lambda r: any(w in r.lower() for w in ['invoice','billing','account','email','settings']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'The mobile app crashes every time I open it on iOS 17.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['crash','bug','fix','team','update','ios']),
         lambda r: sentence_count(r) <= 6]},
    {'ticket': 'I forgot which email address I used to sign up.',
     'checks': [
         lambda r: any(w in r.lower() for w in ['email','account','contact','support','verify']),
         lambda r: sentence_count(r) <= 6]},
] + [
    {'ticket': f'Support ticket #{i}: I need help with my account.',
     'checks': [
         lambda r: len(r.strip()) > 20,
         lambda r: sentence_count(r) <= 8]}
    for i in range(20)
]

REGRESSION_TESTS = REGRESSION_TESTS[:30]   # cap at 30

def run_regression(model, tok, tests):
    results = []
    for t in tests:
        reply  = generate_reply(model, tok, t['ticket'])
        passed = [c(reply) for c in t['checks']]
        results.append({'ticket': t['ticket'], 'reply': reply,
                        'passed': passed, 'all_pass': all(passed)})
    pass_rate = sum(r['all_pass'] for r in results) / len(results)
    return results, pass_rate

print('Running regression suite on base model ...')
base_reg,  base_pass_rate  = run_regression(base_model,  tokenizer, REGRESSION_TESTS)
print('Running regression suite on tuned model ...')
tuned_reg, tuned_pass_rate = run_regression(tuned_model, tokenizer, REGRESSION_TESTS)

print(f'Regression pass rate: base={base_pass_rate*100:.0f}%  '
       f'tuned={tuned_pass_rate*100:.0f}%')
print(f'Deploy gate (>= 90%): base={"GO" if base_pass_rate >= 0.9 else "NO-GO"}  '
       f'tuned={"GO" if tuned_pass_rate >= 0.9 else "NO-GO"}')

## Step 12 — Save Regression Suite as Frozen Test Set

In [ ]:
# Save tickets only (not lambdas — those stay in code)
reg_path = DATA_PROC / 'regression_suite.jsonl'
reg_records = [{'ticket': t['ticket']} for t in REGRESSION_TESTS]
with open(reg_path, 'w') as f:
    for r in reg_records:
        f.write(json.dumps(r) + '\n')
print(f'Regression suite saved: {reg_path} ({len(reg_records)} tests)')
print('IMPORTANT: this file is now frozen. Do not add examples after initial creation.')

## Step 13 — Assemble Scorecard

In [ ]:
base_within_pct  = sum(within_spec(r) for r in base_replies)  / len(base_replies)  * 100
tuned_within_pct = sum(within_spec(r) for r in tuned_replies) / len(tuned_replies) * 100

judge_base_mean  = f'{np.mean(base_judge_scores):.2f}'  if base_judge_scores  else 'N/A'
judge_tuned_mean = f'{np.mean(tuned_judge_scores):.2f}' if tuned_judge_scores else 'N/A'

rows = [
    ('ROUGE-L mean',              f'{np.mean(base_rouge):.3f}',     f'{np.mean(tuned_rouge):.3f}'),
    ('Token F1 mean',             f'{np.mean(base_f1):.3f}',        f'{np.mean(tuned_f1):.3f}'),
    ('Within 2-4 sentences %',    f'{base_within_pct:.0f}%',        f'{tuned_within_pct:.0f}%'),
    ('LLM judge absolute (1-5)',   judge_base_mean,                   judge_tuned_mean),
    ('Hallucination rate %',      f'{base_halluc_rate*100:.0f}%',   f'{tuned_halluc_rate*100:.0f}%'),
    ('Regression pass rate',      f'{base_pass_rate*100:.0f}%',     f'{tuned_pass_rate*100:.0f}%'),
    ('Deploy gate',
     'GO' if base_pass_rate  >= 0.9 else 'NO-GO',
     'GO' if tuned_pass_rate >= 0.9 else 'NO-GO'),
]

print(f'{"Metric":<30} {"Base":>12} {"Tuned":>12}')
print('-' * 56)
for metric, base_val, tuned_val in rows:
    print(f'{metric:<30} {base_val:>12} {tuned_val:>12}')

## Step 14 — Save Eval Scorecard

In [ ]:
lines = [
    '# DeskMate Eval Scorecard — Tuned vs Base\n',
    f'| Metric | Base (Qwen2.5-1.5B) | Tuned (1.5B QLoRA) |',
    f'|---|---|---|',
]
for metric, base_val, tuned_val in rows:
    lines.append(f'| {metric} | {base_val} | {tuned_val} |')

deploy_verdict = 'tuned' if tuned_pass_rate >= 0.9 else 'neither'
lines += [
    '',
    '## Deployment Decision\n',
    f'Deploy gate (≥90% regression pass rate): **{deploy_verdict.upper()}** qualifies.',
    '',
    '## Notes\n',
    '- Hallucination rate uses n-gram entity check; upgrade to NLI in Module 4.4.',
    '- LLM-as-judge scores use both-ordering de-bias; N/A if RUN_JUDGE=False.',
    '- Regression suite frozen at first run — never add examples post-creation.',
]

sc_path = REPORTS / 'eval_scorecard.md'
sc_path.write_text('\n'.join(lines))
print(f'Scorecard saved: {sc_path}')
print()
deploy_model = 'TUNED' if tuned_pass_rate >= 0.9 else ('BASE' if base_pass_rate >= 0.9 else 'NEITHER')
print(f'==> DEPLOY RECOMMENDATION: {deploy_model}')

## Checkpoint

> **Name two failure modes of LLM-as-judge and how you'd mitigate them.**

Write your answer below.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Eval scorecard | `reports/eval_scorecard.md` |
| Regression suite | `data/processed/regression_suite.jsonl` |

**What you've built:** a four-layer eval harness with a hard deploy gate.
Every future model change must pass the regression suite (≥90%) before shipping.

**Next:** Module 3.7 — DPO preference tuning (optional).
When SFT quality is good but tone/safety needs nudging without full retraining.